## MosKita — YOLOv8 Training
### Dengue Mosquito Breeding Site Detection

This notebook trains a YOLOv8s object detection model to identify *Aedes aegypti* / *Aedes albopictus* breeding containers.
The dataset is annotated in Roboflow and exported in YOLOv8 format with 10 classes (Phase 1).

**Hardware:** Lenovo Legion 5 — RTX 2060 6GB, Ryzen 7 4800H, 16GB RAM

**Workflow:**
1. Environment & GPU check
2. Global configuration
3. Dataset verification
4. Training
5. Results & metrics visualization
6. Model export (ONNX / TFLite)

---
### 1 · Environment Setup

In [3]:
import os
import sys
import time
import random
import yaml
import glob

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter

# Attempt to import ultralytics and YOLO; install if missing
try:
    from ultralytics import YOLO
    import ultralytics
except ModuleNotFoundError:
    print("\u26a0\ufe0f  'ultralytics' package not found. Installing via pip...")
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    except Exception as e:
        print(f"\u26a0\ufe0f  Failed to install ultralytics: {e}")
        raise
    # Retry import after installation
    from ultralytics import YOLO
    import ultralytics

print("\u2705 All libraries imported successfully!")
print(f"   PyTorch version      : {torch.__version__}")
print(f"   Ultralytics version  : {ultralytics.__version__}")
print(f"   CUDA available       : {torch.cuda.is_available()}")
print(f"   Device count         : {torch.cuda.device_count()}")
print(f"   Python               : {sys.version.split()[0]}")


⚠️  'ultralytics' package not found. Installing via pip...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 2.4 MB/s  0:00:30m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 2.4 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ultralytics] [ultralytics]n]
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/k1taru/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All libraries imported successfully!
   PyTorch version      : 2.10.0+cu128
   Ultralytics version  : 8.4.38
   CUDA available       : True
   Device count         : 1
   Python               :

#### Detect GPU Available, Details, CUDA, and cuDNN

In [4]:
print("=" * 50)
print("\U0001f5a5\ufe0f  GPU INFORMATION")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_cap  = torch.cuda.get_device_capability(0)
    print(f"\u2705 GPU Detected         : {gpu_name}")
    print(f"   \u2022 Compute Capability : {gpu_cap[0]}.{gpu_cap[1]}")
    print(f"   \u2022 Total VRAM         : {gpu_mem:.2f} GB")
    print(f"   \u2022 VRAM Allocated     : {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    print(f"   \u2022 VRAM Reserved      : {torch.cuda.memory_reserved(0) / (1024**3):.2f} GB")
    DEVICE = 0
    # With fixed input size (640×640), benchmark mode selects the fastest cuDNN kernels
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # determinism handled via SEED in config
else:
    print("\u26a0\ufe0f  No GPU detected — training will use CPU (very slow)")
    DEVICE = "cpu"

print("=" * 50)
print()
print("=" * 50)
print("\u26a1 CUDA / PYTORCH INFORMATION")
print("=" * 50)
print(f"\u2705 CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   \u2022 PyTorch CUDA Ver.  : {torch.version.cuda}")
    print(f"   \u2022 cuDNN Benchmark    : {torch.backends.cudnn.benchmark}")
print(f"   \u2022 PyTorch Version    : {torch.__version__}")
print(f"\u2705 cuDNN Version        : {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else 'N/A'}")
print("=" * 50)


🖥️  GPU INFORMATION
✅ GPU Detected         : NVIDIA GeForce RTX 2060
   • Compute Capability : 7.5
   • Total VRAM         : 5.60 GB
   • VRAM Allocated     : 0.00 GB
   • VRAM Reserved      : 0.00 GB

⚡ CUDA / PYTORCH INFORMATION
✅ CUDA Available       : True
   • PyTorch CUDA Ver.  : 12.8
   • cuDNN Benchmark    : True
   • PyTorch Version    : 2.10.0+cu128
✅ cuDNN Version        : 91002


---
### 2 · Global Configuration

In [5]:
# ============================
# GLOBAL CONFIGURATION
# ============================

# Paths (relative to notebooks/ directory)
DATA_YAML   = "../data/data.yaml"            # YOLOv8 dataset config
PROJECT_DIR = "../models/runs"                # training output directory
EXPORT_DIR  = "../models/exports"             # ONNX export directory

# ============================
# MODEL CONFIGURATION
# ============================
MODEL_VARIANT   = "yolov8s.pt"               # YOLOv8 small — best for RTX 2060 6GB
RUN_NAME        = "moskita_v1"               # name for this training run
EXIST_OK        = False                       # set True to overwrite an existing run

# ============================
# TRAINING HYPERPARAMETERS
# ============================
EPOCHS          = 100                         # max training epochs
BATCH_SIZE      = 16                          # fits RTX 2060 6GB VRAM
IMG_SIZE        = 640                         # YOLOv8 default input resolution
PATIENCE        = 15                          # early stopping patience
OPTIMIZER       = "AdamW"                     # optimizer
LR0             = 0.001                       # initial learning rate
LRF             = 0.01                        # final LR as fraction of lr0
WEIGHT_DECAY    = 0.0005                      # L2 regularization
WARMUP_EPOCHS   = 3.0                         # warmup duration (epochs)
WARMUP_MOMENTUM = 0.8                         # initial momentum during warmup
CLOSE_MOSAIC    = 10                          # disable mosaic aug for final N epochs (aids convergence)
WORKERS         = 4                           # DataLoader worker processes
SAVE_PERIOD     = -1                          # save checkpoint every N epochs (-1 = disabled)

# ============================
# AUGMENTATION CONFIGURATION
# ============================
MOSAIC          = 1.0                         # mosaic augmentation probability
MIXUP           = 0.0                         # mixup augmentation probability
COPY_PASTE      = 0.0                         # copy-paste augmentation probability
FLIPUD          = 0.0                         # vertical flip probability
FLIPLR          = 0.5                         # horizontal flip probability
HSV_H           = 0.015                       # hue shift range
HSV_S           = 0.7                         # saturation shift range
HSV_V           = 0.4                         # value (brightness) shift range
DEGREES         = 10.0                        # rotation range (degrees)
TRANSLATE       = 0.1                         # translation range (fraction)
SCALE           = 0.5                         # scaling range (fraction)
SHEAR           = 2.0                         # shear range (degrees)
PERSPECTIVE     = 0.0                         # perspective distortion

# ============================
# INFERENCE CONFIGURATION
# (used at prediction / export time only — NOT passed to model.train())
# ============================
CONF_THRESHOLD  = 0.4                         # detection confidence threshold
IOU_THRESHOLD   = 0.45                        # NMS IoU threshold

# ============================
# MISC
# ============================
CACHE           = False                       # cache images in RAM (True if ≥32GB RAM)
SEED            = 42                          # reproducibility seed

# Set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("\u2705 Global configuration set successfully!")
print(f"   Model          : {MODEL_VARIANT}")
print(f"   Run name       : {RUN_NAME}")
print(f"   Image size     : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size     : {BATCH_SIZE}")
print(f"   Max epochs     : {EPOCHS}  (early stop patience={PATIENCE})")
print(f"   Close mosaic   : last {CLOSE_MOSAIC} epochs")
print(f"   Optimizer      : {OPTIMIZER} (lr0={LR0}, lrf={LRF})")
print(f"   Workers        : {WORKERS}")
print(f"   Device         : {DEVICE}")


✅ Global configuration set successfully!
   Model          : yolov8s.pt
   Run name       : moskita_v1
   Image size     : 640x640
   Batch size     : 16
   Max epochs     : 100  (early stop patience=15)
   Close mosaic   : last 10 epochs
   Optimizer      : AdamW (lr0=0.001, lrf=0.01)
   Workers        : 4
   Device         : 0


---
### 3 · Dataset Verification

Verify the dataset structure, class distribution, and annotation integrity before training.

In [6]:
# Load and display data.yaml
print("\U0001f50d Loading dataset configuration...")
print()

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

NUM_CLASSES = data_cfg["nc"]
CLASS_NAMES = [data_cfg["names"][i] for i in range(NUM_CLASSES)]

print(f"\u2705 Dataset config loaded: {DATA_YAML}")
print(f"   Number of classes : {NUM_CLASSES}")
print(f"   Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(f"     {i}: {name}")
print()
print(f"   Train images : {data_cfg.get('train', 'N/A')}")
print(f"   Val images   : {data_cfg.get('val', 'N/A')}")
print(f"   Test images  : {data_cfg.get('test', 'N/A')}")

🔍 Loading dataset configuration...

✅ Dataset config loaded: ../data/data.yaml
   Number of classes : 10
   Classes:
     0: plastic_drum_open
     1: plastic_drum_covered
     2: metal_drum_open
     3: discarded_tire_pooled
     4: discarded_tire_dry
     5: flower_pot_saucer_wet
     6: flower_pot_saucer_dry
     7: tarpaulin_pooled
     8: uncovered_container_wet
     9: uncovered_container_dry

   Train images : train/images
   Val images   : val/images
   Test images  : test/images


In [7]:
# Verify dataset split counts & annotation integrity
print("\U0001f4ca DATASET SUMMARY")
print("=" * 70)

# Resolve the dataset root from data.yaml's 'path' field.
# Convention: 'path' is relative to the workspace root (MosKita/).
_yaml_path_val = data_cfg.get("path", ".")
if os.path.isabs(_yaml_path_val):
    dataset_root = Path(_yaml_path_val)
else:
    # data.yaml is one level inside the workspace root (data/data.yaml)
    _workspace_root = Path(DATA_YAML).resolve().parent.parent
    dataset_root = (_workspace_root / _yaml_path_val).resolve()
dataset_root = str(dataset_root)

splits = ["train", "val", "test"]
split_stats = {}

for split in splits:
    img_dir   = os.path.join(dataset_root, split, "images")
    label_dir = os.path.join(dataset_root, split, "labels")

    if os.path.exists(img_dir):
        images = [f for f in os.listdir(img_dir)
                  if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    else:
        images = []

    if os.path.exists(label_dir):
        labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]
    else:
        labels = []

    split_stats[split] = {"images": len(images), "labels": len(labels)}
    status = "\u2705" if len(images) > 0 and len(images) == len(labels) else "\u26a0\ufe0f"
    print(f"   {status} {split:6s} — {len(images):5d} images, {len(labels):5d} labels")

total_images = sum(s["images"] for s in split_stats.values())
total_labels = sum(s["labels"] for s in split_stats.values())

print("=" * 70)
print(f"   Total: {total_images} images, {total_labels} labels")
print(f"   Dataset root: {dataset_root}")

if total_images == 0:
    print()
    print("\u26a0\ufe0f  No images found! Please add your Roboflow export to data/annotated/")
    print("   Expected structure:")
    print("     data/annotated/train/images/  &  data/annotated/train/labels/")
    print("     data/annotated/val/images/    &  data/annotated/val/labels/")
    print("     data/annotated/test/images/   &  data/annotated/test/labels/")


📊 DATASET SUMMARY
   ⚠️ train  —     0 images,     0 labels
   ⚠️ val    —     0 images,     0 labels
   ⚠️ test   —     0 images,     0 labels
   Total: 0 images, 0 labels
   Dataset root: /home/k1taru/Workspace/K1taru/MosKita/data/annotated

⚠️  No images found! Please add your Roboflow export to data/annotated/
   Expected structure:
     data/annotated/train/images/  &  data/annotated/train/labels/
     data/annotated/val/images/    &  data/annotated/val/labels/
     data/annotated/test/images/   &  data/annotated/test/labels/


In [8]:
# Per-class annotation distribution (across all splits)
print("\U0001f4ca PER-CLASS ANNOTATION DISTRIBUTION")
print("=" * 70)

class_counts = Counter()

for split in splits:
    label_dir = os.path.join(dataset_root, split, "labels")
    if not os.path.exists(label_dir):
        continue
    for label_file in os.listdir(label_dir):
        if not label_file.endswith(".txt"):
            continue
        filepath = os.path.join(label_dir, label_file)
        with open(filepath, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:  # class_id + 4 bbox coords minimum
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1

if class_counts:
    print(f"{'Class ID':<10} {'Class Name':<30} {'Annotations':>12}")
    print("-" * 55)
    for i in range(NUM_CLASSES):
        count = class_counts.get(i, 0)
        print(f"{i:<10} {CLASS_NAMES[i]:<30} {count:>12}")
    print("-" * 55)
    print(f"{'Total':<41} {sum(class_counts.values()):>12}")
else:
    print("   No annotations found — dataset is empty or not yet exported.")

📊 PER-CLASS ANNOTATION DISTRIBUTION
   No annotations found — dataset is empty or not yet exported.


In [9]:
# Visualize class distribution (bar chart)
if class_counts:
    fig, ax = plt.subplots(figsize=(12, 5))

    counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]
    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

    bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor="black", linewidth=0.5)

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(count), ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title("MosKita — Per-Class Annotation Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("Class", fontsize=11)
    ax.set_ylabel("Number of Annotations", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Skipping visualization — no annotations found.")

⚠️  Skipping visualization — no annotations found.


In [10]:
# Preview sample training images (if available)
train_img_dir = os.path.join(dataset_root, "train", "images")

if os.path.exists(train_img_dir):
    sample_images = [f for f in os.listdir(train_img_dir)
                     if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if sample_images:
        num_samples = min(8, len(sample_images))
        selected = random.sample(sample_images, num_samples)

        cols = 4
        rows = (num_samples + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
        axes = axes.flatten() if num_samples > 1 else [axes]

        for idx, img_name in enumerate(selected):
            img_path = os.path.join(train_img_dir, img_name)
            img = mpimg.imread(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(img_name[:30], fontsize=8)
            axes[idx].axis("off")

        # Hide unused subplots
        for idx in range(num_samples, len(axes)):
            axes[idx].axis("off")

        plt.suptitle("Sample Training Images", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("\u26a0\ufe0f  No images found in train/images/ — add your Roboflow export first.")
else:
    print("\u26a0\ufe0f  Train images directory not found.")

⚠️  No images found in train/images/ — add your Roboflow export first.


---
### 4 · Model Training

Fine-tune YOLOv8s on the MosKita dataset. Training outputs (weights, plots, metrics) are saved to `models/runs/`.

In [11]:
# Load pretrained YOLOv8s
print("\U0001f680 Loading pretrained model...")
model = YOLO(MODEL_VARIANT)
print(f"\u2705 Model loaded: {MODEL_VARIANT}")
print(f"   Architecture : YOLOv8s (small)")
print(f"   Task         : Object Detection")

🚀 Loading pretrained model...
✅ Model loaded: yolov8s.pt
   Architecture : YOLOv8s (small)
   Task         : Object Detection


In [12]:
# Train
print("\U0001f3cb\ufe0f  Starting training...")
print("=" * 50)

_t_start = time.perf_counter()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project=PROJECT_DIR,
    exist_ok=EXIST_OK,
    patience=PATIENCE,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOMENTUM,
    close_mosaic=CLOSE_MOSAIC,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    flipud=FLIPUD,
    fliplr=FLIPLR,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    cache=CACHE,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    save_period=SAVE_PERIOD,
    plots=True,
    verbose=True,
)

_elapsed = time.perf_counter() - _t_start
_h, _rem = divmod(int(_elapsed), 3600)
_m, _s   = divmod(_rem, 60)

print()
print("\u2705 Training complete!")
print(f"   Elapsed time : {_h:02d}h {_m:02d}m {_s:02d}s")
if torch.cuda.is_available():
    print(f"   Peak VRAM    : {torch.cuda.max_memory_allocated(0) / (1024**3):.2f} GB")


🏋️  Starting training...
Ultralytics 8.4.38 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2060, 5731MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=moskita_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

RuntimeError: Dataset '../data/data.yaml' error ❌ Dataset '../data/data.yaml' images not found, missing path '/home/k1taru/Workspace/K1taru/MosKita/notebooks/datasets/data/annotated/val/images'
Note dataset download directory is '/home/k1taru/Workspace/K1taru/MosKita/notebooks/datasets'. You can update this in '/home/k1taru/.config/Ultralytics/settings.json'

---
### 5 · Results & Metrics Visualization

Display training curves, confusion matrix, and per-class performance from the completed run.

In [ ]:
# Locate the training run output directory
run_dir = os.path.join(PROJECT_DIR, RUN_NAME)

# If run was auto-incremented (e.g. moskita_v12), find the latest
if not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(PROJECT_DIR, f"{RUN_NAME}*")))
    run_dir = candidates[-1] if candidates else run_dir

print(f"\U0001f4c2 Training run directory: {run_dir}")
print()

# List available output files
if os.path.exists(run_dir):
    for item in sorted(os.listdir(run_dir)):
        print(f"   {item}")
else:
    print("\u26a0\ufe0f  Run directory not found — training may not have completed.")

In [ ]:
# Display training results summary
results_csv = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    _map50_col     = "metrics/mAP50(B)"
    _map5095_col   = "metrics/mAP50-95(B)"
    _precision_col = "metrics/precision(B)"
    _recall_col    = "metrics/recall(B)"

    _missing = [c for c in [_map50_col, _map5095_col, _precision_col, _recall_col]
                if c not in df.columns]
    if _missing:
        print(f"\u26a0\ufe0f  Unexpected CSV columns. Available: {list(df.columns)}")
    else:
        best_idx = df[_map50_col].idxmax()
        best_row = df.iloc[best_idx]

        print("\U0001f3c6 TRAINING RESULTS SUMMARY")
        print("=" * 50)
        print(f"   Total epochs      : {len(df)}")
        print(f"   Best epoch        : {int(best_row['epoch']) + 1}")
        print(f"   Best mAP@50       : {best_row[_map50_col]:.4f}")
        print(f"   Best mAP@50-95    : {best_row[_map5095_col]:.4f}")
        print(f"   Best Precision    : {best_row[_precision_col]:.4f}")
        print(f"   Best Recall       : {best_row[_recall_col]:.4f}")

        # Box loss at best epoch
        for loss_col in ["train/box_loss", "val/box_loss"]:
            if loss_col in df.columns:
                print(f"   {loss_col:<22}: {best_row[loss_col]:.4f}")
        print("=" * 50)

        # Target assessment
        map50 = best_row[_map50_col]
        if map50 > 0.85:
            level = "\U0001f31f PUBLISHABLE"
        elif map50 > 0.75:
            level = "\u2705 GOOD"
        elif map50 > 0.60:
            level = "\U0001f44d ACCEPTABLE"
        else:
            level = "\u26a0\ufe0f  BELOW TARGET — consider more data or tuning"
        print(f"   Performance level : {level}")
else:
    print("\u26a0\ufe0f  results.csv not found — training may not have completed.")


In [ ]:
# Training curves
results_png = os.path.join(run_dir, "results.png")

if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  results.png not found.")

In [ ]:
# Confusion matrix
confusion_png = os.path.join(run_dir, "confusion_matrix.png")
confusion_norm_png = os.path.join(run_dir, "confusion_matrix_normalized.png")

plot_files = []
if os.path.exists(confusion_png):
    plot_files.append(("Confusion Matrix", confusion_png))
if os.path.exists(confusion_norm_png):
    plot_files.append(("Confusion Matrix (Normalized)", confusion_norm_png))

if plot_files:
    fig, axes = plt.subplots(1, len(plot_files), figsize=(10 * len(plot_files), 10))
    if len(plot_files) == 1:
        axes = [axes]

    for ax, (title, path) in zip(axes, plot_files):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Confusion matrix plots not found.")

In [ ]:
# Precision-Recall and F1 curves
pr_plots = [
    ("Precision-Recall Curve", os.path.join(run_dir, "PR_curve.png")),
    ("F1 Score Curve", os.path.join(run_dir, "F1_curve.png")),
    ("Precision Curve", os.path.join(run_dir, "P_curve.png")),
    ("Recall Curve", os.path.join(run_dir, "R_curve.png")),
]

available = [(t, p) for t, p in pr_plots if os.path.exists(p)]

if available:
    cols = min(2, len(available))
    rows = (len(available) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 8 * rows))
    axes = np.array(axes).flatten() if len(available) > 1 else [axes]

    for idx, (title, path) in enumerate(available):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].axis("off")
        axes[idx].set_title(title, fontsize=13, fontweight="bold")

    for idx in range(len(available), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  PR / F1 curve plots not found.")

In [ ]:
# Validation batch predictions (visual check)
val_preds = [
    ("Val Batch 0 — Labels", os.path.join(run_dir, "val_batch0_labels.jpg")),
    ("Val Batch 0 — Predictions", os.path.join(run_dir, "val_batch0_pred.jpg")),
]

available_val = [(t, p) for t, p in val_preds if os.path.exists(p)]

if available_val:
    fig, axes = plt.subplots(1, len(available_val), figsize=(12 * len(available_val), 10))
    if len(available_val) == 1:
        axes = [axes]

    for ax, (title, path) in zip(axes, available_val):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Validation prediction images not found.")

---
### 6 · Model Export (ONNX / TFLite)

Export the best model weights for Raspberry Pi 5 edge deployment.

In [ ]:
# Load best weights from the training run
best_weights = os.path.join(run_dir, "weights", "best.pt")

if os.path.exists(best_weights):
    print(f"\U0001f4e6 Loading best weights: {best_weights}")
    best_model = YOLO(best_weights)
    print("\u2705 Best model loaded successfully!")
else:
    print("\u26a0\ufe0f  best.pt not found — using last trained model instead.")
    last_weights = os.path.join(run_dir, "weights", "last.pt")
    if os.path.exists(last_weights):
        best_model = YOLO(last_weights)
        print(f"\u2705 Last model loaded: {last_weights}")
    else:
        best_model = model
        print("\u26a0\ufe0f  No saved weights found — using current model in memory.")

In [ ]:
import shutil

# Export to ONNX (dynamic batch axis, simplified graph)
print("\U0001f680 Exporting to ONNX...")
onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    dynamic=True,        # enables variable batch size at inference time
)
print(f"\u2705 ONNX exported: {onnx_path}")

# Copy to exports directory
os.makedirs(EXPORT_DIR, exist_ok=True)
onnx_dest = os.path.join(EXPORT_DIR, "moskita.onnx")
shutil.copy2(onnx_path, onnx_dest)
print(f"\u2705 Copied to: {onnx_dest}")


In [ ]:
# Export to TFLite (for Pi 5 lightweight inference)
print("\U0001f680 Exporting to TFLite...")
try:
    tflite_path = best_model.export(format="tflite", imgsz=IMG_SIZE)
    print(f"\u2705 TFLite exported: {tflite_path}")

    tflite_dest = os.path.join(EXPORT_DIR, "moskita.tflite")
    shutil.copy2(tflite_path, tflite_dest)
    print(f"\u2705 Copied to: {tflite_dest}")
except Exception as e:
    print(f"\u26a0\ufe0f  TFLite export failed: {e}")
    print("   TFLite export requires tensorflow. Install with: pip install tensorflow")
    print("   You can export on the Pi 5 itself or install TF on this machine.")

---
### 7 · Quick Inference Test

Run a quick inference on a sample image to verify the trained model works.

In [ ]:
# High-risk classes (wet / open containers most likely to harbor dengue vectors)
HIGH_RISK = {
    "plastic_drum_open", "metal_drum_open", "discarded_tire_pooled",
    "flower_pot_saucer_wet", "tarpaulin_pooled", "uncovered_container_wet",
}

# Find a sample test image
test_img_dir = os.path.join(dataset_root, "test", "images")
val_img_dir  = os.path.join(dataset_root, "val", "images")

sample_path = None
for search_dir in [test_img_dir, val_img_dir, train_img_dir]:
    if os.path.exists(search_dir):
        candidates = [f for f in os.listdir(search_dir)
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if candidates:
            sample_path = os.path.join(search_dir, random.choice(candidates))
            break

if sample_path:
    print(f"\U0001f50e Running inference on: {os.path.basename(sample_path)}")
    print()

    inf_results = best_model(sample_path, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)

    # Display results
    for r in inf_results:
        annotated = r.plot()  # BGR numpy array with bounding boxes
        annotated_rgb = annotated[:, :, ::-1]  # convert BGR → RGB for matplotlib

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(annotated_rgb)
        ax.axis("off")
        ax.set_title("MosKita — Sample Detection", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

        # Print detection details
        if r.boxes is not None and len(r.boxes) > 0:
            print(f"\U0001f4cb Detections: {len(r.boxes)}")
            print(f"{'Class':<30} {'Confidence':>10} {'Risk':>12}")
            print("-" * 55)
            for box in r.boxes:
                cls_name = CLASS_NAMES[int(box.cls)]
                conf = float(box.conf)
                risk = "HIGH RISK" if cls_name in HIGH_RISK else "MONITOR"
                print(f"{cls_name:<30} {conf:>10.4f} {risk:>12}")
        else:
            print("No detections found in this image.")
else:
    print("\u26a0\ufe0f  No sample images available for inference test.")
    print("   Add images to your dataset first, then re-run this cell.")

---
### 8 · Training Summary & Next Steps

**What was done:**
1. Trained YOLOv8s on the MosKita dengue breeding site dataset
2. Evaluated performance with mAP, precision, recall, confusion matrix
3. Exported model to ONNX and TFLite for Pi 5 deployment

**Next steps:**
- Run `03_evaluation.ipynb` for detailed per-class analysis
- If mAP@50 < 0.75, apply the **active learning loop** (see Docs/MOSKITA_CONTEXT.md §16)
- Deploy ONNX/TFLite model to Raspberry Pi 5 with `deploy/pi_inference.py`

**Target Metrics:**

| Metric | Acceptable | Good | Publishable |
|---|---|---|---|
| mAP@50 | >0.60 | >0.75 | >0.85 |
| Precision | >0.65 | >0.78 | >0.88 |
| Recall | >0.60 | >0.75 | >0.83 |
| Inference (Pi 5) | <500ms | <200ms | <100ms |